# Guéant–Lehalle–Fernandez-Tapia Market Making Model and Grid Trading

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `GLFT Market Making Model and Grid Trading.ipynb` (`glft_market_making_model_and_grid_trading`).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context('0804T004')
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('glft_market_making_model_and_grid_trading', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

## Overview

Grid trading is straightforward and easy to comprehend, and it excels in high-frequency environments. However, given the intricacies of high-frequency trading, which necessitate comprehensive tick-by-tick simulation with latencies and order fill simulation, optimizing the ideal spread, order interval, and skew can be a challenging task. Furthermore, these values fluctuate over time, especially in response to market conditions, making a fixed setup less than optimal.

To improve grid trading's adaptability, one solution is to combine it with a well-developed market-making model. Let's delve into how this can be achieved.

## Guéant–Lehalle–Fernandez-Tapia Market Making Model

This model represents an advanced evolution of the well-known Avellaneda-Stoikov model and provides a closed-form approximation of asymptotic behavior for terminal time T. Simply, this model does not specify a terminal time, which makes it suitable for typical stocks, spot assets, or crypto perpetual contracts. By employing this model, it is anticipated that the half spread and skew will be accurately adjusted according to market conditions.

In this analysis, we will focus on equations (4.6) and (4.7) in [Optimal market making](https://arxiv.org/abs/1605.01862) and explore how they can be applied to real-world scenarios.

The optimal bid quote depth, $\delta^{b*}_{approx}$, and ask quote depth, $\delta^{a*}_{approx}$, are derived from the fair price as follows:

\begin{align}
\delta^{b*}_{approx}(q) = {1 \over {\xi \Delta}}log(1 + {\xi \Delta \over k}) + {{2q + \Delta} \over 2}\sqrt{{{\gamma \sigma^2} \over {2A\Delta k}}(1 + {\xi \Delta \over k})^{{k \over {\xi \Delta}} + 1}} \label{eq4.6}\tag{4.6} \\
\delta^{a*}_{approx}(q) = {1 \over {\xi \Delta}}log(1 + {\xi \Delta \over k}) - {{2q - \Delta} \over 2}\sqrt{{{\gamma \sigma^2} \over {2A\Delta k}}(1 + {\xi \Delta \over k})^{{k \over {\xi \Delta}} + 1}} \label{eq4.7}\tag{4.7}
\end{align}

Let's introduce $c_1$ and $c_2$ and define them by extracting the volatility 𝜎 from the square root:

\begin{align}
c_1 = {1 \over {\xi \Delta}}log(1 + {\xi \Delta \over k}) \\
c_2 = \sqrt{{\gamma \over {2A\Delta k}}(1 + {\xi \Delta \over k})^{{k \over {\xi \Delta}} + 1}}
\end{align}

Now we can rewrite equations (4.6) and (4.7) as follows:

\begin{align}
\delta^{b*}_{approx}(q) = c_1 + {\Delta \over 2} \sigma c_2 + q \sigma c_2 \\
\delta^{a*}_{approx}(q) = c_1 + {\Delta \over 2} \sigma c_2 - q \sigma c_2
\end{align}

As you can see, this consists of the half spread and skew. $q$ represents a market maker's inventory(position).

\begin{align}
\text{half spread} = C_1 + {\Delta \over 2} \sigma C_2 \\
\text{skew} = \sigma C_2 \\
\delta^{b*}_{approx}(q) = \text{half spread} + \text{skew} \times q \\
\delta^{a*}_{approx}(q) = \text{half spread} - \text{skew} \times q
\end{align}

Thus,

\begin{align}
\text{bid price} = \text{fair price} - (\text{half spread} + \text{skew} \times q) \\
\text{ask price} = \text{fair price} + (\text{half spread} - \text{skew} \times q)
\end{align}

You can find similarities in what the following two articles describe.  
[Stochastic Control Theory and High Frequency Trading](https://ieor.columbia.edu/files/seasdepts/industrial-engineering-operations-research/pdf-files/Borden_D_FESeminar_Sp10.pdf)  
[How to Market Make Bitcoin Derivatives Lesson 2](https://blog.bitmex.com/how-to-market-make-bitcoin-derivatives-lesson-2/)

## Calculating Trading Intensity

To determine the optimal quotes, we need to compute $c_1$ and $c_2$. In order to do that, we need to calibrate $A$ and $k$ of trading intensity, as well as calculate the market volatility $\sigma$.

Trading intensity is defined as:

$$ \lambda = A \exp (-k \delta) $$

We will calibrate these values using market data according to [the this article](https://quant.stackexchange.com/questions/36073/how-does-one-calibrate-lambda-in-a-avellaneda-stoikov-market-making-problem). In order to do that, we need to record market order's arrivals.

Our market maker will react every 100ms, which means they will post or cancel orders at this interval. So, our quotes' trading intensity will be measured in the same time-step. Ideally, we should also account for our orders' queue position; however, to simplify the problem, we will not consider the order queue position in this analysis.

Since we're not considering the order's queue position when measuring trading intensity, only market trades that cross our quote will be counted as executed.

<div class="alert alert-info">

**Note:** The trading intensity in `out` of `measure_trading_intensity` is incorrectly in half-tick units instead of tick units. Although this fix requires adjusting parameters in all related examples, the example is left unchanged to preserve existing results.

</div>

Run HftBacktest to replay the market and record order arrival depth and price changes.

Measure trading intensity from the recorded order arrival depth and plot it.

Calibrate $A$ and $k$ using linear regression, since by taking the logarithm of both sides of lambda, it becomes $log \lambda = -k \delta + logA$.

As you can see, the fitted lambda function is not accurate across the entire range. More specifically, it overestimates the trading intensity for the shallow range near the mid-price and underestimates it for the deep range away from the mid-price.

Since our quotes are likely to be placed in the range close to the mid-price, at least under typical market conditions (excluding high volatility conditions), we will refit the function specifically for the nearest range.

Now, we have a more accurate trading intensity function. Let's see where our quote will be placed.

But before we do that, let's calculate the volatility first.

Compute $c_1$ and $c_2$ according to the equations.

In the Guéant–Lehalle–Fernandez-Tapia formula, $\Delta = 1$ and $\xi = \gamma$. the value of $\gamma$ is arbitrarily chosen.

What does it mean when your quote is positioned 20 ticks away from the mid-price? By analyzing the recorded order arrival depth, you can identify the number of market trades you'll participate in as a market maker, measured in terms of count instead of volume. Additionally, the skew appears to be quite strong, as accumulating just two positions offsets the entire half spread.

Approximately 1.86% of market trades per given time-step could execute your quote. Be aware that it's not the percentage of the traded quantity.

## Implement a Market Maker using the Model


<div class="alert alert-info">
    
**Note:** This example is for educational purposes only and demonstrates effective strategies for high-frequency market-making schemes. All backtests are based on a 0.005% rebate, the highest market maker rebate available on Binance Futures. See <a href="https://www.binance.com/en/support/announcement/binance-updates-usd%E2%93%A2-margined-futures-liquidity-provider-program-2024-06-03-fefc6aa25e0947e2bf745c1c56bea13e">Binance Upgrades USDⓢ-Margined Futures Liquidity Provider Program</a> for more details.
    
</div>

In this example, we will disregard the forecast term and assume that the fair price is equal to the mid price, as we can expect the intrinsic value to remain stable in the short term.

## Adjustment factors

It looks like the skew is too strong, which is why the market maker is hesitant to take on the position. To alleviate the skew, you can introduce adjustment factors, $adj_1$ and $adj_2$, to the calculated half spread and skew, as follow.

$$
\text{half spread}_{adj} = \text{half spread} \times adj_1 \\
\text{skew}_{adj} = \text{skew} \times adj_2
$$

Improved, but even when accounting for rebates, it can only achieve breakeven at best. As shown below, both the half spread and skew move together, primarily influenced by the $c_2$ and the market volatility.

In the 5-day backtest, it's evident that profits are generated through rebates, as a result of maintaining high trading volume by consistently posting quotes.

## Integrating Grid Trading

Creating a grid from the bid and ask prices derived from the Guéant–Lehalle–Fernandez-Tapia market making model.

You can see it works even better with other coins as well. In the next example, we will show how to create multiple markets to achieve better risk-adjusted returns.

## Wrapping up

Thus far, we have illustrated how to apply the model to a real-world example.

For a more effective market-making algorithm, consider dividing this model into the following categories:

* Half-spread: As shown, the half-spread is a function of trading intensity and market volatility. An exponential function used for trading intensity might not be suitable for the entire range. You could develop a more refined approach to convert trading intensity to half-spread. Additionally, while historical trading intensity and market volatility are utilized here, you could forecast short-term trading intensity and volatility to respond more agilely to changes in market conditions. This might involve strategies that use news, events, liquidity vacuums, and other factors to predict volatility explosions.


* Skew: The skew is also a function of trading intensity and market volatility. In this model, only inventory risk is considered, but you can also account for other risks, particularly when making multiple markets. BARRA is a good example of other risks that can be managed similarly.


* Fair Value Pricing: In this model, the fair price is equal to the mid-price, however, you need to incorporate forecasts such as the micro-price and fair value pricing through correlated assets to enhance the strategy.


* Hedging: Hedging is especially crucial when making multiple markets, as it serves as a valuable tool for managing risks.

We will address a few more topics in upcoming examples.

## References
[Dealing with the Inventory Risk - A solution to the market making problem](https://arxiv.org/abs/1105.3115)  
[Optimal market making](https://arxiv.org/abs/1605.01862)  

Knight Capital Group  
[Stochastic Control Theory and High Frequency Trading](https://ieor.columbia.edu/files/seasdepts/industrial-engineering-operations-research/pdf-files/Borden_D_FESeminar_Sp10.pdf)  

BitMEX Market Making Series  
[Algo Trading & Market Making](https://blog.bitmex.com/wp-content/uploads/2019/11/Algo-Trading-and-Market-Making.pdf)  
[How to Market Make Bitcoin Derivatives Lesson 1](https://blog.bitmex.com/how-to-market-make-bitcoin-derivatives-lesson-1/)  
[How to Market Make Bitcoin Derivatives Lesson 2](https://blog.bitmex.com/how-to-market-make-bitcoin-derivatives-lesson-2/)